# Unsupervised Learning — Clustering & Dimensionality Reduction

**Unsupervised learning** finds patterns in data **without labels** — the algorithm discovers structure on its own.

## Two Core Techniques

| Technique | Goal | Algorithm |
|---|---|---|
| **K-Means Clustering** | Group similar data points together | `KMeans` |
| **Dimensionality Reduction** | Compress many features into fewer | `PCA` |

## K-Means Clustering

- Divides data into `k` groups (clusters) based on feature similarity
- Each point belongs to the cluster whose **centroid** (centre) is closest
- **Challenge**: you must pick `k` in advance → use the **Elbow Method** to find the best `k`
- **Sensitive to scale** → always scale features before clustering

## PCA (Principal Component Analysis)

- Reduces many correlated features into fewer **principal components**
- Each component captures a percentage of the total information in the original data
- Use when: too many features, features are correlated, or you want to visualise high-dimensional data in 2D/3D

In [ ]:
## ── K-MEANS CLUSTERING ───────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Mall customer dataset: Age and Annual Spending Score (1–100)
np.random.seed(42)
n = 150
data = {
    'CustomerID': range(1, n + 1),
    'Age': np.concatenate([
        np.random.normal(25, 5, 50),   # young shoppers
        np.random.normal(45, 5, 60),   # middle-aged
        np.random.normal(65, 5, 40),   # seniors
    ]).clip(18, 80).astype(int),
    'Spending': np.concatenate([
        np.random.normal(75, 10, 50),  # high spenders
        np.random.normal(50, 12, 60),  # moderate spenders
        np.random.normal(28, 8,  40),  # low spenders
    ]).clip(1, 100).astype(int),
}
df = pd.DataFrame(data)
X = df[['Age', 'Spending']]

print('=== Dataset Overview ===')
print(f'{df.shape[0]} customers')
print(df.describe().round(1).to_string())

# ── SCALE FEATURES ────────────────────────────────────────────────────────────
# K-Means uses Euclidean distance → scaling is critical
# Without it, Age (18–80 range) would visually dominate Spending (1–100 range)
# after scaling both features have mean≈0 and std≈1, so neither dominates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── ELBOW METHOD — Find the best K ───────────────────────────────────────────
# inertia_ = total sum of squared distances from each point to its nearest centroid
# As K increases, inertia always decreases (more clusters → tighter groups)
# We want the "elbow" — the K where improvement starts to plateau (diminishing returns)
inertias = []
k_range  = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

print('\n=== Elbow Method — Inertia per K ===')
for k, iner in zip(k_range, inertias):
    relative_bar = '█' * int(iner / inertias[0] * 25)
    print(f'  K={k:2d}  inertia={iner:7.1f}  {relative_bar}')
print('  → Look for the K where the bar length stops dropping sharply (the elbow)')

# ── FIT FINAL MODEL ───────────────────────────────────────────────────────────
k_chosen = 3  # elbow is at K=3 for this dataset
model = KMeans(n_clusters=k_chosen, random_state=42, n_init=10)
df['Cluster'] = model.fit_predict(X_scaled)  # assigns cluster ID (0, 1, or 2) to each row

# Cluster centres in ORIGINAL (unscaled) space for interpretability
centres_original = scaler.inverse_transform(model.cluster_centers_)
centres_df = pd.DataFrame(centres_original, columns=['Avg Age', 'Avg Spending'])
centres_df.index.name = 'Cluster'
print('\n=== Cluster Centroids (original scale) ===')
print(centres_df.round(1))

# ── CLUSTER LABELING ──────────────────────────────────────────────────────────
# Give business-meaningful names to each cluster based on centroid values
# Sort by Average Spending (low → high) and assign labels accordingly
sorted_by_spend = centres_df['Avg Spending'].sort_values().index.tolist()
labels = ['Budget Shoppers', 'Moderate Shoppers', 'Premium Shoppers']
label_map = {cluster_id: label for cluster_id, label in zip(sorted_by_spend, labels)}
df['Segment'] = df['Cluster'].map(label_map)

print('\n=== Cluster Summary (count, mean age, mean spending) ===')
summary = df.groupby('Segment')[['Age', 'Spending']].agg(['mean', 'count'])
print(summary.round(1))

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('K-Means Clustering — Mall Customer Segmentation', fontsize=13, fontweight='bold')

# Plot 1: Elbow Curve
axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', lw=2)
axes[0].axvline(k_chosen, color='red', linestyle='--', lw=2, label=f'Chosen K={k_chosen}')
axes[0].scatter([k_chosen], [inertias[k_chosen - 1]], color='red', s=130, zorder=5)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (total within-cluster variance)')
axes[0].set_title('Elbow Method\n(pick K at the bend — diminishing returns)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Clusters coloured by segment
color_map = {
    'Budget Shoppers':   'tomato',
    'Moderate Shoppers': 'steelblue',
    'Premium Shoppers':  'mediumseagreen',
}
for segment, colour in color_map.items():
    mask = df['Segment'] == segment
    axes[1].scatter(df.loc[mask, 'Age'], df.loc[mask, 'Spending'],
                    color=colour, label=segment, s=60, alpha=0.8)

# Plot centroids as black X markers
axes[1].scatter(centres_original[:, 0], centres_original[:, 1],
                marker='X', s=220, color='black', zorder=10, label='Centroids')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Spending Score')
axes[1].set_title(f'K-Means Clusters  (K={k_chosen})\ncolours = customer segments')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

# Plot 3: Bar chart — how many customers are in each segment?
seg_counts = df['Segment'].value_counts()
bars = axes[2].bar(seg_counts.index, seg_counts.values,
                   color=[color_map[s] for s in seg_counts.index], edgecolor='black')
axes[2].bar_label(bars, label_type='edge', fontsize=11, padding=3)
axes[2].set_title('Customers per Segment')
axes[2].set_ylabel('Number of Customers')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'\nK-Means converged in {model.n_iter_} iterations')
print(f'Final inertia: {model.inertia_:.2f}')
print(f'n_init=10 means K-Means was run 10 times with different random starts → best result kept')

In [ ]:
## ── PCA — PRINCIPAL COMPONENT ANALYSIS ──────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Higher-dimensional dataset — PCA is most useful when you have MANY correlated features
np.random.seed(42)
n = 200
df_pca = pd.DataFrame({
    'Age':         np.random.randint(18, 70, n),
    'Income':      np.random.randint(20000, 120000, n),
    'Spending':    np.random.randint(1, 100, n),
    'Savings':     np.random.randint(1000, 50000, n),
    'CreditScore': np.random.randint(300, 850, n),
    'LoanAmount':  np.random.randint(0, 500000, n),
})

print('=== Original Dataset ===')
print(f'{df_pca.shape[0]} rows × {df_pca.shape[1]} features')
print(df_pca.describe().round(0).to_string())

# ── SCALE FIRST ────────────────────────────────────────────────────────────────
# PCA is sensitive to feature magnitude — Income (20k–120k) would completely
# dominate Age (18–70) unless we scale. After StandardScaler, all features
# have mean=0 and std=1, so PCA treats each feature equally.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca)

# ── PCA: Reduce 6 dimensions → 2 ─────────────────────────────────────────────
# n_components=2 keeps only the top 2 principal components
# Each component is a linear combination of ALL original features,
# chosen to capture the maximum remaining variance
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)  # shape: (200, 2) instead of (200, 6)

ev = pca.explained_variance_ratio_  # fraction of total variance in each PC
print(f'\n=== PCA: 6D → 2D ===')
print(f'PC1 captures : {ev[0]*100:.1f}% of total variance')
print(f'PC2 captures : {ev[1]*100:.1f}% of total variance')
print(f'Total retained: {sum(ev)*100:.1f}%  ({(1-sum(ev))*100:.1f}% discarded)')

# components_: each ROW = one PC = weights on each original feature
# High absolute weight → that feature contributes heavily to this component
print('\n=== Component Loadings (which features drive each PC?) ===')
loadings = pd.DataFrame(
    pca.components_.T,
    index=df_pca.columns,
    columns=['PC1', 'PC2']
)
print(loadings.round(3))

# ── FULL VARIANCE ANALYSIS (choose how many PCs to keep) ─────────────────────
# Rule of thumb: keep enough PCs to explain 80–90% of total variance
pca_full = PCA()
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

print('\n=== Cumulative Variance by Number of Components ===')
for i, (ev_i, cv) in enumerate(zip(pca_full.explained_variance_ratio_, cumvar), 1):
    bar = '█' * int(cv * 20)
    print(f'  {i} PCs → {cv*100:.1f}% cumulative  {bar}')

n_for_80 = next(i for i, c in enumerate(cumvar, 1) if c >= 0.80)
print(f'\n  → Need {n_for_80} PCs to explain ≥80% of variance  (vs {df_pca.shape[1]} original features)')

# ── VISUALISATION ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('PCA — Principal Component Analysis', fontsize=13, fontweight='bold')

# Plot 1: Scree Plot — how much variance each PC holds
bars = axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
                   pca_full.explained_variance_ratio_ * 100,
                   color='steelblue', edgecolor='black', alpha=0.8)
axes[0].plot(range(1, len(cumvar) + 1), cumvar * 100, 'ro-', lw=2, label='Cumulative %')
axes[0].axhline(80, color='green', linestyle='--', lw=1.5, label='80% threshold')
axes[0].set_xlabel('Principal Component Number')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot\n(how much information each PC holds)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: 2D Projection — the full dataset viewed in reduced space
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, color='mediumseagreen', s=40)
axes[1].set_xlabel(f'PC1  ({ev[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2  ({ev[1]*100:.1f}% variance)')
axes[1].set_title(f'6D Data Projected to 2D\n({sum(ev)*100:.1f}% of information kept)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Loading Plot — which features drive each PC
# Arrows pointing in similar directions = features are correlated
# Long arrows = feature has strong influence on PCs
for i, feature in enumerate(df_pca.columns):
    axes[2].arrow(0, 0,
                  pca.components_[0, i],
                  pca.components_[1, i],
                  head_width=0.03, head_length=0.03, fc='tomato', ec='tomato')
    axes[2].text(pca.components_[0, i] * 1.15,
                 pca.components_[1, i] * 1.15,
                 feature, fontsize=9, ha='center')
axes[2].set_xlim(-1.3, 1.3)
axes[2].set_ylim(-1.3, 1.3)
axes[2].axhline(0, color='black', lw=0.8)
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_xlabel('PC1 Loading')
axes[2].set_ylabel('PC2 Loading')
axes[2].set_title('Feature Loading Plot\n(arrow direction = how each feature contributes)')
axes[2].grid(True, alpha=0.3)
circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', color='gray', alpha=0.5)
axes[2].add_patch(circle)

plt.tight_layout()
plt.show()

print('\n=== Unsupervised Learning Quick Reference ===')
print('''
K-Means:
  km = KMeans(n_clusters=k, n_init=10, random_state=42)
  km.fit_predict(X_scaled)       → cluster IDs (0 to k-1)
  km.cluster_centers_            → centroid coordinates (scaled space)
  km.inertia_                    → total within-cluster squared distance
  Elbow method: plot inertia for k=1..10, choose K at the "elbow"

PCA:
  pca = PCA(n_components=2)
  pca.fit_transform(X_scaled)    → compressed data matrix
  pca.explained_variance_ratio_  → % variance captured per component
  pca.components_                → feature weights (loadings) per component
  Rule: keep enough PCs so that cumulative variance ≥ 80–90%

Key rule for both: ALWAYS scale features first (StandardScaler)
''')

## DBSCAN Clustering (Density-Based)

**K-Means limitation:** assumes clusters are round (circular) and you must know K in advance.

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) solves both problems:
- Finds clusters of **any shape** (elongated, crescent, irregular)
- Automatically determines the number of clusters
- Identifies **noise points** (outliers) — labeled as `-1`

**Two key parameters:**
- `eps` — the radius of the neighborhood around each point (how close = "neighbor")
- `min_samples` — minimum neighbors needed to be a **core point** (dense region)

| Point type | Meaning |
|---|---|
| **Core point** | Has ≥ `min_samples` neighbors within `eps` |
| **Border point** | Within `eps` of a core point, but fewer than `min_samples` neighbors |
| **Noise point** | Not within `eps` of any core point — labeled `-1` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs

np.random.seed(42)

# make_moons creates two interleaved crescent shapes — K-Means cannot handle this
# DBSCAN detects density-connected regions and handles non-circular shapes well
X_moons, _ = make_moons(n_samples=200, noise=0.08, random_state=42)

# Scale before DBSCAN (eps is distance-based)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_moons)

# ── DBSCAN ────────────────────────────────────────────────────────────────────
db = DBSCAN(eps=0.3, min_samples=5)
labels_db = db.fit_predict(X_scaled)  # -1 = noise, 0,1,2... = cluster IDs

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()

print('=== DBSCAN Results ===')
print(f'eps={db.eps}, min_samples={db.min_samples}')
print(f'Clusters found : {n_clusters}  (no need to specify K in advance)')
print(f'Noise points   : {n_noise}  (labeled -1 — outliers DBSCAN rejected)')
for cid in sorted(set(labels_db)):
    label = f'Cluster {cid}' if cid != -1 else 'Noise (-1)'
    print(f'  {label:<14}: {(labels_db==cid).sum()} points')

# ── K-Means on same data (fails on non-circular shapes) ──────────────────────
km = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)

# ── Effect of eps on DBSCAN ───────────────────────────────────────────────────
print('\n=== Effect of eps (min_samples=5) ===')
print(f'{"eps":<8} {"Clusters":>10} {"Noise pts":>12}')
print('-' * 32)
for eps in [0.1, 0.2, 0.3, 0.5, 0.8, 1.5]:
    lbs = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
    nc  = len(set(lbs)) - (1 if -1 in lbs else 0)
    nn  = (lbs == -1).sum()
    print(f'{eps:<8} {nc:>10} {nn:>12}')
print('→ Small eps = too many noise points  |  Large eps = merges everything into 1 cluster')

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('DBSCAN vs K-Means on Non-Circular Data', fontsize=13, fontweight='bold')

colors_db = ['tomato' if l == 0 else 'steelblue' if l == 1 else 'gray'
             for l in labels_db]
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=colors_db, s=40, alpha=0.8)
axes[0].set_title(f'DBSCAN (eps=0.3, min_samples=5)\n{n_clusters} clusters found, {n_noise} noise points')
axes[0].grid(True, alpha=0.3)

colors_km = ['tomato' if l == 0 else 'steelblue' for l in labels_km]
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=colors_km, s=40, alpha=0.8)
axes[1].set_title('K-Means (k=2)\nFails on crescent shapes — wrong boundaries')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Hierarchical Clustering (Agglomerative)

Builds a **tree of clusters** (called a dendrogram) by repeatedly merging the two closest clusters.

- **Agglomerative (bottom-up):** starts with each point as its own cluster, merges them up
- No need to choose K upfront — you can cut the dendrogram at any height to get any K
- `linkage` controls how "distance between clusters" is measured:
  - `'ward'` — minimises total within-cluster variance (most common, produces compact clusters)
  - `'complete'` — max distance between any two points in the clusters
  - `'average'` — mean distance between all pairs

**Dendrogram reading:** the height where two branches join = the distance at which those clusters merged. A large jump in height = natural cut point → that's a good K.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)

# 3 well-separated blobs — hierarchical clustering should find all 3
X, y_true = make_blobs(n_samples=60, centers=3, cluster_std=0.8, random_state=42)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── AgglomerativeClustering (sklearn) ────────────────────────────────────────
model = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels = model.fit_predict(X_scaled)

print('=== Agglomerative Clustering (ward linkage, n_clusters=3) ===')
for cid in range(3):
    print(f'  Cluster {cid}: {(labels == cid).sum()} points')

# ── Compare linkage methods ────────────────────────────────────────────────────
print('\n=== Effect of linkage method ===')
from sklearn.metrics import adjusted_rand_score
for linkage_method in ['ward', 'complete', 'average', 'single']:
    m = AgglomerativeClustering(n_clusters=3, linkage=linkage_method)
    lbs = m.fit_predict(X_scaled)
    ari = adjusted_rand_score(y_true, lbs)   # 1.0 = perfect match to true labels
    print(f'  {linkage_method:<10}: ARI={ari:.4f}  (1.0 = perfect)')

# ── Dendrogram — visualise the full merge history ─────────────────────────────
# scipy's linkage() computes the full merge tree
# condensed_distances + method='ward' matches AgglomerativeClustering(linkage='ward')
Z = linkage(X_scaled, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hierarchical (Agglomerative) Clustering', fontsize=13, fontweight='bold')

# Plot 1: Dendrogram
# Y-axis = distance at which clusters merged (higher = merged later = more different)
# Horizontal dashed line = where you would "cut" to get 3 clusters
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=15,
           leaf_rotation=90, leaf_font_size=9, color_threshold=4.5)
axes[0].axhline(y=4.5, color='red', linestyle='--', lw=2,
                label='Cut here → 3 clusters')
axes[0].set_title('Dendrogram\n(red line = cut to get 3 clusters)')
axes[0].set_xlabel('Sample index (or cluster size in brackets)')
axes[0].set_ylabel('Merge distance (ward)')
axes[0].legend()

# Plot 2: Cluster assignments scatter
colors = ['tomato', 'steelblue', 'mediumseagreen']
for cid in range(3):
    mask = labels == cid
    axes[1].scatter(X[mask, 0], X[mask, 1],
                    color=colors[cid], label=f'Cluster {cid}',
                    s=60, alpha=0.8)
axes[1].set_title('Cluster Assignments (ward, k=3)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\n=== Clustering Methods Quick Comparison ===')
summary = pd.DataFrame({
    'Method':      ['K-Means', 'DBSCAN', 'Hierarchical'],
    'Need K?':     ['Yes', 'No', 'Flexible (cut dendrogram)'],
    'Cluster shape': ['Circular only', 'Any shape', 'Any shape'],
    'Handles noise':  ['No', 'Yes (-1 label)', 'No'],
    'Best for':    ['Round clusters, large data', 'Irregular shapes, outliers', 'Small data, exploring K'],
})
print(summary.to_string(index=False))